# LLM Inference Benchmark — Qwen2.5-7B on A100-SXM4-80GB

Measures prefill and decode separately and checks both against the hardware roofline.

| | |
|---|---|
| Model | Qwen2.5-7B-Instruct (bf16, 7.6B params, 14.25 GiB weights) |
| GPU | 1x NVIDIA A100-SXM4-80GB (sm80, 2039 GB/s, 312 TFLOP/s bf16) |
| Engine | vLLM 0.7.3, torch 2.5.1+cu124, `enforce_eager=True`, TP=1 |
| KV cache | 1,038,128 tokens (~127x concurrency at 8192 tokens/request) |

See `README.md` for the analysis. `bench.py` is the standalone script.

In [ ]:
import time
from pathlib import Path
import numpy as np, pandas as pd
from vllm import LLM, SamplingParams
from vllm.inputs import TokensPrompt

MODEL, MAX_LEN = "Qwen/Qwen2.5-7B-Instruct", 8192
PARAMS, HIDDEN, LAYERS, KV_HEADS, HEAD_DIM = 7.615e9, 3584, 28, 4, 128
PEAK_BW, PEAK_FLOPS = 2039e9, 312e12

t0 = time.perf_counter()
llm = LLM(model=MODEL, tensor_parallel_size=1, dtype="bfloat16",
          max_model_len=MAX_LEN, gpu_memory_utilization=0.90,
          enforce_eager=True, seed=1234)
print(f"engine ready in {time.perf_counter()-t0:.0f}s")

for path in ("llm_engine.cache_config", "llm_engine.model_config.cache_config"):
    o = llm
    try:
        for k in path.split("."):
            o = getattr(o, k)
        b = getattr(o, "num_gpu_blocks", None)
        if b:
            KV = b * o.block_size
            print(f"KV cache: {KV:,} tokens -> {KV//MAX_LEN}x concurrency at max length")
            break
    except AttributeError:
        continue

## Measurement method

vLLM's offline `generate()` returns one final result per prompt, so there is no per-token stream to time directly. TTFT and ITL are obtained by subtraction: one run asking for 1 token measures prefill alone, a second run with N tokens measures prefill + decode. The prompt is identical in both runs, so the prefill cost cancels.

This is the reason the `itl_p50` / `itl_p99` columns from earlier drafts of this work were invalid — without per-token timestamps there are no percentiles to compute. Only mean ITL is reported here.

In [ ]:
def exact_prompt(n, seed=0):
    rng = np.random.default_rng(seed)
    return TokensPrompt(prompt_token_ids=rng.integers(1000, 150000, n).tolist())

def measure(prompt, max_tokens, seed=0):
    t0 = time.perf_counter()
    llm.generate([prompt], SamplingParams(max_tokens=1, temperature=0.0, ignore_eos=True))
    ttft_ms = (time.perf_counter() - t0) * 1e3

    t0 = time.perf_counter()
    out = llm.generate([prompt], SamplingParams(max_tokens=max_tokens,
                                                temperature=0.0, ignore_eos=True))
    total_ms = (time.perf_counter() - t0) * 1e3
    n_out = len(out[0].outputs[0].token_ids)

    decode_ms = total_ms - ttft_ms
    return dict(prompt_tokens=len(out[0].prompt_token_ids), output_tokens=n_out,
                ttft_ms=ttft_ms, decode_ms=decode_ms,
                itl_mean_ms=decode_ms / max(n_out - 1, 1),
                decode_tok_s=(n_out - 1) / (decode_ms / 1e3))

## Experiment 1 — prefill: TTFT vs prompt length

Prompts are token-exact (`TokensPrompt`), not string repetition, so the x-axis is the real token count. Output length is 1, so there is no decode phase and ITL is undefined — recorded as NaN.

In [ ]:
OUT = Path("results"); OUT.mkdir(exist_ok=True)

rows = []
for n in [128, 256, 512, 1024, 2048, 4096, 8192]:
    r = measure(exact_prompt(n), 1)
    r.update(decode_ms=np.nan, itl_mean_ms=np.nan, decode_tok_s=np.nan)
    rows.append(r)
    print(f"n={n:5d}  TTFT={r['ttft_ms']:7.1f} ms  {r['ttft_ms']*1e6/n:6.0f} ns/token")
prefill = pd.DataFrame(rows)
prefill.to_csv(OUT / "prefill.csv", index=False)

## Experiment 2 — decode: ITL vs generation length

Prompt pinned at 512 tokens so prefill cost is constant across the sweep. If ITL is flat, decode cost per token does not depend on how much has already been generated.

In [ ]:
rows = []
for n in [16, 64, 128, 256, 512]:
    r = measure(exact_prompt(512), n)
    rows.append(r)
    print(f"out={n:4d}  ITL={r['itl_mean_ms']:6.2f} ms  {r['decode_tok_s']:6.1f} tok/s")
decode = pd.DataFrame(rows)
decode.to_csv(OUT / "decode.csv", index=False)

## Experiment 3 — throughput vs concurrency

Distinct prompt per slot. Identical prompts would be served from the prefix cache and would measure nothing about real scheduling.

In [ ]:
rows = []
for c in [1, 2, 4, 8, 16, 32, 64]:
    prompts = [exact_prompt(512, seed=200+i) for i in range(c)]
    t0 = time.perf_counter()
    out = llm.generate(prompts, SamplingParams(max_tokens=256,
                                                temperature=0.0, ignore_eos=True),
                       use_tqdm=False)
    wall = time.perf_counter() - t0
    toks = sum(len(o.outputs[0].token_ids) for o in out)
    rows.append(dict(concurrency=c, total_tokens=toks, wall_s=wall,
                     aggregate_tok_s=toks/wall, per_user_tok_s=toks/wall/c))
    print(f"c={c:3d}  aggregate={toks/wall:8.1f} tok/s  per-user={toks/wall/c:6.2f}")
throughput = pd.DataFrame(rows)
throughput.to_csv(OUT / "throughput.csv", index=False)

## Roofline analysis

Decode has arithmetic intensity ~1 FLOP/byte: each weight is read from HBM once per token and used for a single multiply-accumulate. The A100 ridge point is 312e12 / 2039e9 = 153 FLOP/byte, so decode sits roughly 150x below it and is firmly bandwidth-bound.

Prefill is the opposite. Weights are amortised across all n prompt tokens, so intensity rises roughly linearly with n (520 FLOP/byte at n=512, 10217 at n=8192) and sits well above the ridge. Attention contributes `2*n^2*d*L` — the factor of 2 is the causal mask, which skips the upper triangle of the score matrix.

In [ ]:
ridge = PEAK_FLOPS / PEAK_BW
print(f"ridge point {ridge:.0f} FLOP/byte; decode intensity ~1.00 "
      f"({100/ridge:.2f}% of ridge -> memory-bound)\n")

for n in (512, 2048, 4096, 8192):
    ttft = prefill.ttft_ms[prefill.prompt_tokens == n].iloc[0] / 1e3
    gemm = 2 * PARAMS * n
    attn = 2 * n * n * HIDDEN * LAYERS
    flops = gemm + attn
    print(f"n={n:5d}  intensity {flops/(2*PARAMS):8.0f}  "
          f"attention {100*attn/flops:5.1f}% of FLOPs  "
          f"MFU {100*flops/ttft/PEAK_FLOPS:5.1f}%")

W = 14.25 * 2**30
itl = decode.itl_mean_ms.iloc[-1] / 1e3
print(f"\nITL roofline {W/PEAK_BW*1e3:.2f} ms ({1/(W/PEAK_BW):.0f} tok/s ceiling)")
print(f"measured     {itl*1e3:.2f} ms ({1/itl:.0f} tok/s)")
print(f"achieved bandwidth {100*(PARAMS*2/itl)/PEAK_BW:.1f}% of peak")

row = throughput[throughput.concurrency == 64].iloc[0]
sps = row.total_tokens / 64 / row.wall_s
kv_b = 64 * (512 + 128) * 2 * LAYERS * KV_HEADS * HEAD_DIM * 2
print(f"\nc=64: {sps:.0f} steps/s, "
      f"{(W+kv_b)*sps/1e9:.0f} GB/s ({100*(W+kv_b)*sps/PEAK_BW:.0f}% of peak), "
      f"{2*PARAMS*64*sps/1e12:.0f} TFLOP/s ({100*2*PARAMS*64*sps/PEAK_FLOPS:.0f}% of peak)")
print("-> neither roof saturated; the limit is per-step overhead")

## Figures

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(prefill.prompt_tokens, prefill.ttft_ms, "o-")
ax[0].axhline(30, ls=":", c="gray", lw=1, label="~30 ms fixed overhead")
ax[0].set(xlabel="prompt tokens", ylabel="TTFT (ms)",
          title="Prefill: TTFT vs context (compute-bound)")
ax[0].legend(fontsize=8); ax[0].set_xscale("log", base=2)
ax[1].plot(decode.output_tokens, decode.itl_mean_ms, "s-")
ax[1].axhline(decode.itl_mean_ms.iloc[-1], ls=":", c="gray", lw=1)
ax[1].set(xlabel="output tokens", ylabel="ITL (ms/token)",
          title="Decode: ITL vs output length (bandwidth-bound)")
ax[1].set_ylim(0, 16)
ax[2].plot(throughput.concurrency, throughput.aggregate_tok_s, "o-", label="aggregate")
ax[2].plot(throughput.concurrency, throughput.per_user_tok_s, "s--", label="per user")
ax[2].axvline(16, ls=":", c="r", lw=1, label="saturation knee")
ax[2].set(xlabel="concurrency", ylabel="tokens/sec", title="Throughput vs concurrency")
ax[2].legend(fontsize=8); ax[2].set_xscale("log", base=2)
plt.tight_layout(); plt.savefig(OUT / "figures.png", dpi=150)
print("saved")

In [ ]:
ridge = PEAK_FLOPS / PEAK_BW
fig, ax = plt.subplots(figsize=(8, 5.5))
ai = np.logspace(0, np.log10(ridge*2), 100)
ax.plot(ai, PEAK_BW*ai, c="k", lw=1.6, label=f"compute roof ({PEAK_FLOPS/1e12:.0f} TFLOP/s)")
ax.plot(ai, np.full_like(ai, PEAK_BW), c="k", ls="--", lw=1.6,
        label=f"memory roof ({PEAK_BW/1e9:.0f} GB/s)")
ax.axvline(ridge, color="gray", ls=":", lw=1)
ax.annotate(f"ridge {ridge:.0f} FLOP/byte", xy=(ridge, 4e13),
            xytext=(ridge*1.3, 4e13), fontsize=8, color="gray")

itl = decode.itl_mean_ms.iloc[-1]/1e3
ax.scatter([2*PARAMS/(2*PARAMS)], [PARAMS*2/itl], s=160, c="red", zorder=5,
           label=f"decode b=1: {itl*1e3:.2f} ms/tok\n{PARAMS*2/itl/1e9:.0f} GB/s "
                 f"= {100*PARAMS*2/itl/PEAK_BW:.0f}% of roof")
for n in [2048, 4096, 8192]:
    ttft = prefill.ttft_ms[prefill.prompt_tokens == n].iloc[0]/1e3
    fl = 2*PARAMS*n + 2*n*n*HIDDEN*LAYERS
    ax.scatter([fl/(2*PARAMS)], [fl/ttft], s=90, c="tab:blue", zorder=5,
               label=f"prefill n={n} ({100*fl/ttft/PEAK_FLOPS:.0f}% MFU)" if n == 2048 else None)
ax.scatter([], [], s=90, c="tab:blue", label="prefill n=4096, 8192")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(0.5, 6e3); ax.set_ylim(1e11, 8e14)
ax.set_xlabel("arithmetic intensity (FLOP/byte)")
ax.set_ylabel("achieved throughput (FLOP/s or byte/s)")
ax.set_title("Roofline — Qwen2.5-7B (bf16) on A100-SXM4-80GB")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "roofline.png", dpi=150, bbox_inches="tight")
print("saved")